# HeartAI — Clinical Model Evaluation & Diagnostic Calibration
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/24abh/HeartAI/blob/main/notebooks/model_evaluation.ipynb)

Comprehensive evaluation of candidate models on the held-out test split. Computes:
- ROC-AUC and Precision-Recall Curves
- Sensitivity (Recall) and Specificity
- Probability calibration (Brier score)
- Feature importance attribution


In [ ]:
# Step 1: Environment & Test Split Ingestion
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, precision_recall_curve, confusion_matrix,
    brier_score_loss, ConfusionMatrixDisplay
)
import joblib

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/HeartAI'
except ImportError:
    DRIVE_DIR = './drive_mock'

test_df = pd.read_csv(f"{DRIVE_DIR}/data/heart_test.csv")
X_test, y_test = test_df.drop(columns=['target']), test_df['target']
print(f"Test evaluation set loaded: {X_test.shape[0]} patients.")


In [ ]:
# Step 2: Benchmark Evaluation Function
def evaluate_clinical_model(name, y_true, y_prob):
    y_pred = (y_prob >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    metrics = {
        'model': name,
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'roc_auc': float(roc_auc_score(y_true, y_prob)),
        'sensitivity_recall': float(tp / (tp + fn)),
        'specificity': float(tn / (tn + fp)),
        'precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'f1_score': float(f1_score(y_true, y_pred)),
        'brier_score': float(brier_score_loss(y_true, y_prob))
    }
    return metrics


In [ ]:
# Step 3: Compute Test Set Metrics
from xgboost import XGBClassifier
train_df = pd.read_csv(f"{DRIVE_DIR}/data/heart_train.csv")
X_train, y_train = train_df.drop(columns=['target']), train_df['target']

xgb = XGBClassifier(n_estimators=120, max_depth=4, learning_rate=0.05, random_state=42)
xgb.fit(X_train, y_train)
y_prob_xgb = xgb.predict_proba(X_test)[:, 1]

results = evaluate_clinical_model('XGBoost-HeartAI', y_test, y_prob_xgb)
print("=== CLINICAL MODEL EVALUATION REPORT ===")
for k, v in results.items():
    if k != 'model':
        print(f"  {k:<20}: {v:.4f}")


In [ ]:
# Step 4: ROC and Precision-Recall Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob_xgb)
axes[0].plot(fpr, tpr, color='#0d9488', lw=2.5, label=f'XGBoost (AUC = {results["roc_auc"]:.3f})')
axes[0].plot([0, 1], [0, 1], color='navy', lw=1.5, linestyle='--')
axes[0].set_xlabel('False Positive Rate (1 - Specificity)')
axes[0].set_ylabel('True Positive Rate (Sensitivity)')
axes[0].set_title('Receiver Operating Characteristic (ROC)')
axes[0].legend(loc="lower right")

# Confusion Matrix
cm = confusion_matrix(y_test, y_prob_xgb >= 0.5)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No CAD', 'CAD Present'])
disp.plot(cmap='Blues', ax=axes[1])
axes[1].set_title('Clinical Confusion Matrix (Test Set)')

plt.tight_layout()
plt.show()
